In [1]:
# Import the necessary Python modules.

# Data Management/Investigation
import pandas as pd
from pandas.api.types import CategoricalDtype # Ordering categories
import requests # For downloading the website
from bs4 import BeautifulSoup # For parsing the website
import numpy as np
import missingno as miss

# Plotting libraries
from plotnine import *
import matplotlib.pyplot as plt

# For pre-processing data 
from sklearn import preprocessing as pp 
from sklearn.compose import ColumnTransformer 

# For splits and CV
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold # Cross validation 
from sklearn.model_selection import cross_validate # Cross validation 
from sklearn.model_selection import GridSearchCV # Cross validation + param. tuning.

# Machine learning methods 
from sklearn.naive_bayes import GaussianNB as NB
from sklearn.neighbors import KNeighborsClassifier as KNN
from sklearn.tree import DecisionTreeClassifier as DT
from sklearn.tree import DecisionTreeRegressor as DT_reg
from sklearn.ensemble import RandomForestClassifier as RF
from sklearn import tree # For plotting the decision tree rules

# For evaluating our model's performance
import sklearn.metrics as m

# Pipeline to combine modeling elements
from sklearn.pipeline import Pipeline

# For model interpretation
from sklearn.inspection import (
    permutation_importance,
    partial_dependence, 
    PartialDependenceDisplay, 
    plot_partial_dependence
)

# For iterating through directory
import os

# Misc
import warnings
warnings.filterwarnings("ignore")

United States Census Bureau’s “Explore Census Data”<br/>
Identify income in the past twelve months by ZIP Code

In [2]:
# Load the Census Data collected from the U.S. Census Bureau.
census_data = pd.read_csv("Census_Data/ACSST5Y2019.S1901_data_with_overlays_2021-11-04T111610.csv", 
                          low_memory = False)

In [4]:
# Look at the datatypes of the Census Data.
census_data.dtypes

GEO_ID            object
NAME              object
S1901_C01_001E    object
S1901_C01_001M    object
S1901_C01_002E    object
                   ...  
S1901_C04_014M    object
S1901_C04_015E    object
S1901_C04_015M    object
S1901_C04_016E    object
S1901_C04_016M    object
Length: 130, dtype: object

In [5]:
# All ZIP Codes in our data.
all_zip_codes = pd.DataFrame((census_data
                              .NAME[1:]
                              .str
                              .replace("ZCTA5 ", "")
                              .unique()
                              .astype("str")
                             ), columns = ["zip_code"])

In [6]:
# Total number of ZIP Codes in CA in our data.
# https://data.ca.gov/dataset/county-and-zip-code-references
ca_zip_codes = pd.read_csv("Census_Data/zip-code-list.csv")

In [8]:
# Convert to object data type.
ca_zip_codes = ca_zip_codes.astype("str")

In [9]:
# Number of zip codes in CA in our data.
len(all_zip_codes
    .merge(ca_zip_codes, how = "inner", on = "zip_code")
    .zip_code
    .unique()
   )

1763

American Hospital Directory (AHD) - List of Hospital Websites and Additional Information

In [10]:
def scrap_AHD():
    """
    Web scraping techniques to collect a list of hospitals in California and push the results into a CSV to be saved.
    
    Arguments:
        None
    
    Return:
        None
    """
    # Prepare list to hold the UN information.
    scraped_data = []

    # AHD website - CA search results for all hospitals.
    url = "https://www.ahd.com/list_cms.php?mstate%5B%5D=CA&listing=1&viewmap=0"

    # Download the webpage.
    page = requests.get(url)

    # If a connection was successfully reached.
    if page.status_code == 200:
        # Parse the webpage.
        soup = BeautifulSoup(page.content, "html.parser")

        # Identify all table rows on the webpage.
        # Ignore the first seven rows.
        table_rows = soup.find_all("tr")[7:485]

        # Iterate through each table row.
        for table_row in table_rows:
            # Identify all cells within the table row.
            table_cell = table_row.find_all("td")

            # Hospital Name.
            name = str(table_cell[0].text)

            # Hospital Beds.
            beds = int(table_cell[1].text)

            # Hospital City.
            city = str(table_cell[2].text)

            # Pull the items in the cell.
            html_soup = BeautifulSoup(str(table_cell))

            # Iterate through the cell and find all a-tags with href.
            for href_url in html_soup.find_all("a", href = True):
                # Pull the href from the table for only the first entry.
                website = "https://www.ahd.com" + href_url["href"]
                break

            # Add row to dataframe.
            scraped_data.append([name, beds, city, website])

    # Convert the holding list to Pandas DataFrame.
    dat = pd.DataFrame(scraped_data, columns = ["name", "beds", "city", "website"])

    # Save the hospital information to a CSV.
    dat.to_csv("Hospital_Data/AHD_list.csv")

In [11]:
# Load the AHD list we scrapped.
ahd_list = pd.read_csv("Hospital_Data/AHD_list.csv")

# Print out the results of scrapping.
print("Number of Hosps with Beds:", len(ahd_list.query("beds > 0")))
print("Number of Hosps with No Beds:", len(ahd_list.query("beds == 0")))

Number of Hosps with Beds: 408
Number of Hosps with No Beds: 70


In [12]:
def read_OSHPD(base_code):
    """
    Reads OSHPD files where files are housed in separate folders for each hospital.
    Looks for pricing data given a HCPCS code and returns results for all hospitals in a Pandas DataFrame.
    
    Arguments:
        HCPCS code of interest to collect pricing data
        
    Return:
        Pandas DataFrame
    """
    
    # Create empty dataframe to hold the OSHPD pricing data.
    final_dt = pd.DataFrame(columns = ["HospitalDirectory"])

    # Directory we will read from to obtain pricing data.
    directory = "Hospital_Data/OSHPD_2019"

    # Iterate through each folder in the directory.
    for subdir, dirs, files in os.walk(directory):
        # Iterate through each file in the folder.
        for file in files:
            # If the file is an Excel file.
            if file[-4:] == "xlsx" or file[-3:] == "xls":
                # Determine the file path of the Excel doc.
                file_path = os.path.join(subdir, file)

                try:
                    # Read the Excel to Pandas.
                    sheet_to_df_map = pd.read_excel(file_path, sheet_name = None)

                    # If there was one sheet loaded from the Excel.
                    if len(sheet_to_df_map.keys()) == 1:
                        # Make sheet into dataframe.
                        df = pd.DataFrame(list(sheet_to_df_map.values())[0])

                        # Check for base_code as a string in any column.
                        entry_base_code_str = df[df.eq(str(base_code)).any(1)]

                        # Check base_code as an integer in any column.
                        entry_base_code_int = df[df.eq(int(base_code)).any(1)]

                        # If the string version is found.
                        if entry_base_code_str.shape[0] > 0:
                            # Add rows to the final dataframe.
                            final_dt = pd.concat([final_dt, entry_base_code_str], sort = False)
                            
                            # Continue to next file.
                            continue

                        # If the int version is found.
                        if entry_base_code_int.shape[0] > 0:
                            # Add rows to the final dataframe.
                            final_dt = pd.concat([final_dt, entry_base_code_int], sort = False)
                    else:
                        # If there was more than one sheet loaded from the Excel, iterate through each sheet.
                        for sheet in sheet_to_df_map.values():
                            # Make sheet into dataframe.
                            df = pd.DataFrame(sheet)

                            # Check base_code as a string in any column.
                            entry_base_code_str = df[df.eq(str(base_code)).any(1)]

                            # Check base_code as an integer in any column.
                            entry_base_code_int = df[df.eq(int(base_code)).any(1)]

                            # If the string version is found.
                            if entry_base_code_str.shape[0] > 0:
                                # Add rows to the final dataframe.
                                final_dt = pd.concat([final_dt, entry_base_code_str], sort = False)
                                
                                # Continue to next file.
                                continue

                            # If the int version is found.
                            if entry_base_code_int.shape[0] > 0:
                                # Add rows to the final dataframe.
                                final_dt = pd.concat([final_dt, entry_base_code_int], sort = False)
                                
                    # Fill the NAs with the considered Hospital Directory.
                    final_dt["HospitalDirectory"] = final_dt["HospitalDirectory"].fillna(subdir)
                except:
                    continue
                    
    # Return the final dataframe.
    return final_dt

In [13]:
# Collect hospital pricing data from OSHPD directory of hospital files.
oshpd_70450 = read_OSHPD(70450)

WARNING *** file size (16320) not 512 + multiple of sector size (512)
WARNING *** file size (2719680) not 512 + multiple of sector size (512)


In [14]:
# Select only the columns of interest to identify the correct pricing.
oshpd_70450_clean = oshpd_70450[["HospitalDirectory", "Unnamed: 1", "Unnamed: 2", "PRICE", 
                     "June 2019 Prices", "Charge Amount", "STD AMOUNT",
                     "Rate", "Amount", "Price", "UNIT CHARGE AMOUNT",
                     "AMOUNT", "Total Price ", "CUR CHG"]]

In [15]:
# Remove spaces in column names.
oshpd_70450_clean.columns = oshpd_70450_clean.columns.map(lambda x: x.replace(' ', '_'))

# Remove colons in column names.
oshpd_70450_clean.columns = oshpd_70450_clean.columns.map(lambda x: x.replace(':', ''))

In [16]:
# Iterate through each pricing column. Work from far right to the left.
for col in reversed(range(2, len(oshpd_70450_clean.columns) - 1)):
    # If the column is NA, then fill with value from the next column.
    oshpd_70450_clean.iloc[:, col].fillna(oshpd_70450_clean.iloc[:, col + 1], inplace = True)

In [17]:
# Create new column named "HospitalPrice".
oshpd_70450_clean.eval("HospitalPrice_70450 = Unnamed_2", inplace = True)

In [18]:
# Collect final cleaned OSHPD data.
oshpd_70450_final = oshpd_70450_clean[["HospitalDirectory", "HospitalPrice_70450"]]

In [19]:
# Convert all results to numeric where all strings will be converted to NAs.
oshpd_70450_final["HospitalPrice_70450"] = pd.to_numeric(oshpd_70450_final.HospitalPrice_70450, errors = "coerce")

In [23]:
# Drop the duplicates for each hospital. Only keep the first entry.
oshpd_70450_final = oshpd_70450_final.drop_duplicates(subset = ["HospitalDirectory"], keep = "first")

In [40]:
# If the price looks like the HCPCS Code 70450, then mark as NA.
oshpd_70450_final.loc[oshpd_70450_final["HospitalPrice_70450"] == 70450, "HospitalPrice_70450"] = np.nan

In [46]:
# Rename hospital directory as hospital name without path.
oshpd_70450_final["HospitalName"] = oshpd_70450_final["HospitalDirectory"].str.split("/").str[-1]

In [51]:
# Save the OSHPD data to a CSV.
oshpd_70450_final.filter(items = ["HospitalName", "HospitalPrice_70450"]).to_csv("Hospital_Data/OSHPD_list.csv")

In [88]:
# Reload the OSHPD pricing data for hospitals in CA with Zip Code.
oshpd_70450_zipcode = pd.read_csv("Hospital_Data/OSHPD_ZipCode_list.csv")

In [89]:
oshpd_70450_zipcode["ZipCode"] = oshpd_70450_zipcode["ZipCode"].astype(str)

In [90]:
# Remove the first row since it was a subheader.
census_data_clean = census_data.iloc[1:, :]

In [91]:
# Clean up the Zip Code entries.
census_data_clean["NAME"] = census_data_clean["NAME"].str.replace("ZCTA5 ", "").astype(str)

In [92]:
# Merge the hospital pricing data with census data.
merged_data = pd.merge(oshpd_70450_zipcode, census_data_clean, how = "left", left_on = "ZipCode", right_on = "NAME")

In [93]:
# Collect the columns of interest we will include in our machine learning pipeline.
final_data = merged_data[["HospitalName", "HospitalPrice_70450", "ZipCode", "S1901_C01_012E", "S1901_C01_013E"]]

In [94]:
# Rname columns to understand what they actually are.
final_data = final_data.rename(columns = {"S1901_C01_012E" : "MedIncome", "S1901_C01_013E" : "MeanIncome"})

In [95]:
# Save the final CSV final to push into our data analysis.
final_data.to_csv("Hospital_Data/Final_Pricing.csv")